# Nifty 50 Swing Bot — Weekly Retrain & Deploy

**Run every Sunday.**  
This notebook:
1. Resolves last week's predictions (actual prices vs predicted)
2. Retrains the model with all historical + new data
3. Compares new model vs currently deployed model
4. Deploys if better → saves to Drive + Supabase
5. Generates next week's signals

**Before running:** Add `SUPABASE_URL` and `SUPABASE_KEY` to Colab Secrets (🔑 icon in left sidebar).

In [ ]:
# ── Cell 1: GPU check + environment setup ──────────────────────────────────
import os
import subprocess

# Confirm GPU
result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — will use CPU (slower training)')

# Load Supabase credentials from Colab Secrets
try:
    from google.colab import userdata
    os.environ['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
    os.environ['SUPABASE_KEY'] = userdata.get('SUPABASE_KEY')
    print('Supabase credentials loaded from Colab Secrets')
except Exception as e:
    print(f'Could not load Colab Secrets: {e}')
    print('Set SUPABASE_URL and SUPABASE_KEY manually below if needed:')
    # os.environ['SUPABASE_URL'] = 'https://xxx.supabase.co'
    # os.environ['SUPABASE_KEY'] = 'your-anon-key'

# Configuration
REPO_DIR   = '/content/swing_bot'
DRIVE_DIR  = '/content/drive/MyDrive/swing_outputs'
HORIZON    = 5       # trading days
TRIALS     = 50      # Optuna trials (reduce to 10 for quick test)
CAPITAL    = 1_000_000   # paper trading capital INR

print(f'Config: horizon={HORIZON}d | trials={TRIALS} | capital=₹{CAPITAL:,.0f}')

In [ ]:
# ── Cell 2: Mount Drive + clone/update repo ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

# Clone or pull repo
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # ← UPDATE THIS

if not Path(REPO_DIR).exists():
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
print(f'Repo ready at {REPO_DIR}')

# Restore previous models from Drive (so --fast-signals can load params)
drive_models = Path(DRIVE_DIR) / 'models'
repo_models  = Path(REPO_DIR) / 'models'
if drive_models.exists():
    shutil.copytree(str(drive_models), str(repo_models), dirs_exist_ok=True)
    print(f'Models restored from Drive → {repo_models}')
else:
    repo_models.mkdir(parents=True, exist_ok=True)
    print('No previous models on Drive — starting fresh')

# Restore portfolio state
drive_outputs = Path(DRIVE_DIR) / 'outputs'
repo_outputs  = Path(REPO_DIR) / 'outputs'
if drive_outputs.exists():
    shutil.copytree(str(drive_outputs), str(repo_outputs), dirs_exist_ok=True)
    print(f'Outputs (portfolio, signals) restored from Drive')

%cd {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)

In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
!pip install -q optuna yfinance supabase streamlit plotly python-dotenv
# Note: pandas-ta skipped on Colab due to numpy 2.x compatibility issues
# The pipeline uses fallback TA implementations in that case.

# Verify XGBoost + GPU
import xgboost as xgb
print(f'XGBoost {xgb.__version__}')
try:
    m = xgb.XGBClassifier(device='cuda', n_estimators=1)
    import numpy as np
    m.fit(np.random.randn(10, 2), np.random.randint(0, 2, 10))
    print('GPU (CUDA) training: AVAILABLE')
except Exception as e:
    print(f'GPU not available ({e}) — using CPU')

In [ ]:
# ── Cell 4: Resolve last week's predictions ──────────────────────────────────
import math
from datetime import date
from src.config import Config
from src.data.ingestion import fetch_prices, fetch_index_prices, UNIVERSE
from src.db.supabase_client import get_supabase_client
from src.tracking.outcome_tracker import resolve_outcomes, compute_recent_ic, compute_weekly_ic_series
from src.models.improvement import load_current_model_ic, should_retrain, get_model_version

sb = get_supabase_client()
print(f'Supabase: {"connected" if sb else "JSON-only mode"}')

# Fetch price data needed to resolve outcomes
print('Fetching price data …')
price_df = fetch_prices(UNIVERSE, start='2024-01-01')  # only recent data needed for resolution
print(f'  {len(price_df):,} rows loaded')

# Resolve
n_resolved = resolve_outcomes(price_df, sb, 'outputs', HORIZON)
print(f'  Resolved {n_resolved} prediction(s)')

# Current deployed model's IC
current_ic, current_run_id = load_current_model_ic(sb)
print(f'  Current deployed model: {current_run_id or "none"}')
print(f'  4-week rolling IC: {current_ic:.4f}' if not math.isnan(current_ic) else '  4-week rolling IC: n/a (insufficient data)')

# IC trend
ic_series = compute_weekly_ic_series(sb, n_weeks=8)
if not ic_series.empty:
    print('\nIC trend (last 8 weeks):')
    print(ic_series.to_string(index=False))

# Emergency check
needs_emergency, msg = should_retrain(current_ic)
if needs_emergency:
    print(f'\n⚠️  EMERGENCY: {msg}')
else:
    print(f'\nModel health: {msg}')

In [ ]:
# ── Cell 5: Full retrain ─────────────────────────────────────────────────────
from src.pipeline.runner import run

new_run_id = get_model_version()
print(f'New run_id: {new_run_id}')

train_cfg = Config(
    horizon           = HORIZON,
    xgb_n_trials      = TRIALS,
    skip_backtest     = False,
    save_outputs      = True,
    save_to_supabase  = True,
    paper_trade       = True,
    initial_capital   = CAPITAL,
    max_positions     = 10,
    model_version     = new_run_id,
    resolve_outcomes_on_start = False,   # already done in Cell 4
    rebalance_every   = HORIZON,
    embargo           = HORIZON,
)

new_stats, new_signals = run(train_cfg)
print(f'\nNew model — OOF IC={new_stats.get("oof_ic", "n/a")}  Sharpe={new_stats.get("Sharpe", "n/a")}  CAGR={new_stats.get("CAGR", "n/a")}')

In [ ]:
# ── Cell 6: Compare new model vs currently deployed ──────────────────────────
from src.models.improvement import compare_models

new_oof_ic = new_stats.get('oof_ic', float('nan'))
if new_oof_ic is None: new_oof_ic = float('nan')

should_deploy = compare_models(float(new_oof_ic), current_ic, min_improvement=0.005)

print(f'Current deployed IC : {"{:.4f}".format(current_ic) if not math.isnan(current_ic) else "n/a"}')
print(f'New model OOF IC    : {"{:.4f}".format(new_oof_ic) if not math.isnan(new_oof_ic) else "n/a"}')
print(f'Min improvement req : 0.0050')
print(f'Deploy decision     : {"✅ YES — deploying new model" if should_deploy else "❌ NO — keeping current model"}')

In [ ]:
# ── Cell 7: Deploy if better ─────────────────────────────────────────────────
import shutil
from pathlib import Path
from src.models.improvement import mark_deployed

if should_deploy:
    # Mark in Supabase + local JSON
    mark_deployed(new_run_id, sb, 'outputs', previous_run_id=current_run_id)

    # Copy models + outputs to Drive
    drive_path = Path(DRIVE_DIR)
    drive_path.mkdir(parents=True, exist_ok=True)
    shutil.copytree('models',  str(drive_path / 'models'),  dirs_exist_ok=True)
    shutil.copytree('outputs', str(drive_path / 'outputs'), dirs_exist_ok=True)
    print(f'Model {new_run_id} deployed and saved to {DRIVE_DIR}')
else:
    # Save new model to Drive as a candidate (for manual review)
    cand_path = Path(DRIVE_DIR) / 'candidates' / new_run_id
    cand_path.mkdir(parents=True, exist_ok=True)
    shutil.copy('models/best_params.json', str(cand_path / 'best_params.json'))
    print(f'Current model kept. New candidate saved to {cand_path}')

In [ ]:
# ── Cell 8: Generate next-week signals (fast mode, no re-training) ───────────
sig_cfg = Config(
    horizon           = HORIZON,
    xgb_n_trials      = 0,          # load saved best_params.json
    skip_backtest     = True,        # skip walk-forward (~18 min), only need signals
    save_outputs      = True,
    save_to_supabase  = True,
    paper_trade       = True,
    initial_capital   = CAPITAL,
    max_positions     = 10,
    model_version     = new_run_id if should_deploy else current_run_id,
    resolve_outcomes_on_start = False,
    rebalance_every   = HORIZON,
    embargo           = HORIZON,
)
_, week_signals = run(sig_cfg)
print(f'\nGenerated {len(week_signals)} signals for next week')
if not week_signals.empty:
    print(week_signals[['ticker','signal','prob_up','entry_price','stop_loss','target_price']].to_string(index=False))

In [ ]:
# ── Cell 9: Summary ─────────────────────────────────────────────────────────
from datetime import date
import math

long_count = int((week_signals.get('signal', '') == 'LONG').sum()) if not week_signals.empty else 0

print('═' * 60)
print('  WEEKLY RETRAIN SUMMARY')
print('═' * 60)
print(f'  Date                  : {date.today()}')
print(f'  New run_id            : {new_run_id}')
print(f'  Outcomes resolved     : {n_resolved}')
print(f'  Previous IC (4-wk)    : {"{:.4f}".format(current_ic) if not math.isnan(current_ic) else "n/a"}')
print(f'  New model OOF IC      : {"{:.4f}".format(float(new_oof_ic)) if not math.isnan(float(new_oof_ic)) else "n/a"}')
print(f'  Deployment decision   : {"DEPLOYED" if should_deploy else "KEPT PREVIOUS"}')
print(f'  Next-week signals     : {len(week_signals)} total  |  {long_count} LONG')
print('─' * 60)
print(f'  View dashboard        : https://your-app.streamlit.app')
print(f'  Next run              : Next Sunday morning')
print('═' * 60)

if not ic_series.empty:
    print('\nIC trend (last 8 weeks):')
    print(ic_series.to_string(index=False))